In [12]:
pip install pyvis

Import the SUBSET.csv into a bipartite network.

In [9]:
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite

# 1. Load your adjacency list file
# Change 'interactions.csv' to your filename. Use sep='\t' if it is a TSV file.
df = pd.read_csv('Adjacency_Filtered.csv')

# 2. Extract unique lists of nodes for each bipartite set
# Assumes your columns are named 'miRNA' and 'mRNA'
mirnas = df['miRNA'].unique()
mrnas = df['Gene'].unique()

# 3. Initialize an empty graph
B = nx.Graph()

# 4. Add nodes with their respective bipartite attribute (0 or 1)
B.add_nodes_from(mirnas, bipartite=0)
B.add_nodes_from(mrnas, bipartite=1)

# 5. Populate edges directly from the DataFrame rows
# This loops through each row and adds an edge between the miRNA and mRNA
edges = list(df.itertuples(index=False, name=None))
B.add_edges_from(edges)

# 6. Verify the import worked correctly
print(f"Graph is bipartite: {bipartite.is_bipartite(B)}")
print(f"Total nodes: {B.number_of_nodes()} ({len(mirnas)} miRNAs, {len(mrnas)} mRNAs)")
print(f"Total interactions: {B.number_of_edges()}")

Graph is bipartite: True
Total nodes: 483 (296 miRNAs, 187 mRNAs)
Total interactions: 1913


Initial visualization

In [15]:
from pyvis.network import Network
#from pyvis.network import Network

# Initialize PyVis network (notebook=True embeds it inline in Jupyter)
net = Network(notebook=True, cdn_resources='in_line', height="600px", width="100%", bgcolor="#222222", font_color="white")

# Load your existing NetworkX graph (B)
net.from_nx(B)

# Apply visual configurations based on attributes
for node in net.nodes:
    # 1. Shape/Color based on molecule type
    if node['bipartite'] == 0:
        node['color'] = '#FF6B6B'  # Salmon for miRNA
        node['shape'] = 'diamond'
    else:
        node['color'] = '#4D96FF'  # Sky Blue for mRNA
        node['shape'] = 'dot'

    # 2. Size nodes by the magnitude of their absolute fold-change
    # Fallback to size 10 if 'fold_change' is missing
    fc = node.get('fold_change', 0)
    node['size'] = 10 + (abs(fc) * 5)

    # 3. Create a hover tooltip showing names and data
    node['title'] = f"ID: {node['id']}<br>Fold Change: {fc:.2f}"

# Turn on physics engine so nodes naturally space themselves out
net.toggle_physics(True)

# Save and display inside the notebook
net.show("mirna_mrna_network.html")

mirna_mrna_network.html


Import gene expression levels (ie. from the dataset 1790Nanostring.csv), and make them become Node attributes for their specific node


In [14]:
import pandas as pd
import networkx as nx

# --- 1. Load your Fold-Change Data ---
# Assumes columns: 'ID' (e.g., mRNA_A) and 'FoldChange'
mrna_fc_df = pd.read_csv('nanostring_mrna.csv')
# Note: 'FoldChange' column does not exist. Assuming 'miR155' is the intended expression data.
# Note: 'ID' column does not exist. Assuming 'ProbeID' is the intended identifier.
mirna_fc_df = pd.read_csv('nanostring_mirna.csv', header=None, names=['ProbeID', 'Accession', 'Analyte', 'Control', 'cel67', 'miR142', 'miR155', 'miR18b', 'miR890'])
# Note: 'FoldChange' column does not exist. Assuming '877.45' is the intended expression data.
# Note: 'ID' column does not exist. Assuming 'hsa-let-7b-5p' is the intended identifier.

# --- 2. Convert DataFrames to Dictionaries ---
# This creates a quick lookup structure: { 'mRNA_A': 2.5, 'mRNA_B': -1.2 }
mrna_fc_dict = pd.Series(mrna_fc_df['miR155'].values, index=mrna_fc_df['ProbeID']).to_dict()
mirna_fc_dict = pd.Series(mirna_fc_df['miR155'].values, index=mirna_fc_df['ProbeID']).to_dict()

# Combine both into a single attribute dictionary
all_fold_changes = {**mrna_fc_dict, **mirna_fc_dict}

# --- 3. Attach Attributes to the NetworkX Nodes ---
nx.set_node_attributes(B, all_fold_changes, name='fold_change')

# --- 4. Verify and Access the Data ---
# Let's check a specific node's data
node_example = mrna_fc_df['ProbeID'].iloc[0] # Changed to 'ProbeID'
print(f"Node: {node_example}")
print(f"Attributes: {B.nodes[node_example]}")

Node: ACTG1
Attributes: {'bipartite': 1, 'fold_change': 1170.03, 'raw_expression': 1170.03, 'size': 10}


In [16]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

# Define your exact experimental condition column
target_condition = 'miR155'

# Load clean pre-filtered mRNA file
mrna_df = pd.read_csv('nanostring_mrna.csv')
mrna_exp_dict = pd.Series(mrna_df[target_condition].values, index=mrna_df['ProbeID']).to_dict()

# Load clean pre-filtered miRNA file
mirna_df = pd.read_csv('nanostring_mirna.csv', header=None, names=['ProbeID', 'Accession', 'Analyte', 'Control', 'cel67', 'miR142', 'miR155', 'miR18b', 'miR890'])

mirna_exp_dict = pd.Series(mirna_df[target_condition].values, index=mirna_df['ProbeID']).to_dict()

# Combine both dictionary structures into one tracking database
all_raw_expressions = {**mrna_exp_dict, **mirna_exp_dict}

# Push clean values onto the loaded NetworkX graph (B)
nx.set_node_attributes(B, all_raw_expressions, name='raw_expression')

net = Network(notebook=True, cdn_resources='in_line', height="600px", width="100%", bgcolor="#222222", font_color="white")
net.from_nx(B)

# Calculate independent maximums from the graph layers to preserve visual parity

max_mirna_val = max([B.nodes[n].get('raw_expression', 1) for n, attr in B.nodes(data=True) if attr.get('bipartite') == 0] or [1])
max_mrna_val = max([B.nodes[n].get('raw_expression', 1) for n, attr in B.nodes(data=True) if attr.get('bipartite') == 1] or [1])

for node in net.nodes:
    raw_exp = node.get('raw_expression', 0)

    # Scale each molecule type relative to its own layer maximum (Base size 10 to Max size 40)
    if node['bipartite'] == 0:
        node['color'] = '#FF6B6B'  # Salmon for miRNA
        node['shape'] = 'diamond'
        node['size'] = 10 + ((raw_exp / max_mirna_val) * 30)
    else:
        node['color'] = '#4D96FF'  # Sky Blue for mRNA
        node['shape'] = 'dot'
        node['size'] = 10 + ((raw_exp / max_mrna_val) * 30)

    # Tooltip tracking windows reveal actual absolute numbers on hover
    node['title'] = f"ID: {node['id']}<br>Raw Expression: {raw_exp:.2f}"

#toggle physics for a moving visualization
net.toggle_physics(True)
net.show("mirna_mrna_network.html")

mirna_mrna_network.html
